# UPI Fraud Ring & Merchant Analytics
## 01 : Data Profiling

Systematic profiling of Transactions, KYC, Merchant Master, and Chargeback datasets.

**Principle:** Profile first, clean later. Do not blindly delete messy records.


In [1]:
# CELL 1: Imports and project configuration
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project Directory :", BASE_DIR.resolve())
print("Raw Data Directory:", RAW_DIR.resolve())
print("Processed Directory:", PROCESSED_DIR.resolve())


Project Directory : C:\Users\HP\Desktop\Datathon
Raw Data Directory: C:\Users\HP\Desktop\Datathon\data\raw
Processed Directory: C:\Users\HP\Desktop\Datathon\data\processed


In [2]:
# CELL 2: Verify raw files
raw_files = sorted(RAW_DIR.iterdir())
print("Files available in raw directory:")
for file in raw_files:
    print("-", file.name)


Files available in raw directory:
- track1_chargebacks.json
- track1_dataset_notes.txt
- track1_kyc_records.csv
- track1_merchants_master.csv
- track1_upi_transactions.csv


In [3]:
# CELL 3: Load all raw datasets
transactions = pd.read_csv(RAW_DIR / "track1_upi_transactions.csv")
kyc = pd.read_csv(RAW_DIR / "track1_kyc_records.csv")
merchants = pd.read_csv(RAW_DIR / "track1_merchants_master.csv")

with open(RAW_DIR / "track1_chargebacks.json", "r", encoding="utf-8") as f:
    chargeback_data = json.load(f)

chargebacks = pd.DataFrame(chargeback_data)

print("Transactions :", transactions.shape)
print("KYC          :", kyc.shape)
print("Merchants    :", merchants.shape)
print("Chargebacks  :", chargebacks.shape)


Transactions : (20400, 8)
KYC          : (36400, 12)
Merchants    : (6210, 11)
Chargebacks  : (2884, 13)


In [4]:
# CELL 4: Dataset overview
datasets = {
    "Transactions": transactions,
    "KYC": kyc,
    "Merchants": merchants,
    "Chargebacks": chargebacks
}

for name, df in datasets.items():
    print(f"{name:15} | Rows: {len(df):,} | Columns: {len(df.columns)}")


Transactions    | Rows: 20,400 | Columns: 8
KYC             | Rows: 36,400 | Columns: 12
Merchants       | Rows: 6,210 | Columns: 11
Chargebacks     | Rows: 2,884 | Columns: 13


In [5]:
# CELL 5: Schema and data types
for name, df in datasets.items():
    print("\n" + "=" * 90)
    print(name.upper())
    print("=" * 90)
    display(pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Non-Null": df.notna().sum().values,
        "Unique": df.nunique(dropna=True).values
    }))



TRANSACTIONS


,Column,Data Type,Non-Null,Unique
0,txn_id,object,20400,20000
1,timestamp,object,20400,19083
2,user_id,object,20400,17878
3,merchant_id,object,20400,8051
4,amount,object,20400,19900
5,utr,object,19376,19000
6,mcc,float64,17474,5
7,status,object,20400,14



KYC


,Column,Data Type,Non-Null,Unique
0,user_id,object,36400,32165
1,full_name,object,36400,33921
2,pan,object,34504,33164
3,aadhaar,object,33736,32093
4,date_of_birth,object,33456,29284
5,city,object,36400,41
6,state,object,36400,9
7,monthly_income,object,33467,26034
8,occupation,object,36400,9
9,signup_timestamp,object,33490,14607



MERCHANTS


,Column,Data Type,Non-Null,Unique
0,merchant_id,object,6210,5083
1,merchant_name,object,6210,5732
2,mcc,object,5696,42
3,merchant_category,object,6210,82
4,business_type,object,6210,14
5,city,object,6210,41
6,state,object,6210,9
7,onboarding_date,object,5711,4399
8,settlement_account,object,3759,3548
9,merchant_status,object,6210,15



CHARGEBACKS


,Column,Data Type,Non-Null,Unique
0,complaint_id,object,2884,2800
1,txn_id,object,2884,2582
2,user_id,object,2884,2454
3,merchant_id,object,2884,2051
4,transaction_timestamp,object,2884,1238
5,reported_timestamp,object,2884,1329
6,disputed_amount,object,2884,2609
7,reason_code,object,2884,34
8,complaint_text,object,2884,84
9,resolution_status,object,2884,13


In [6]:
# CELL 6: Missing value audit
for name, df in datasets.items():
    print("\n" + "=" * 90)
    print(f"{name.upper()} — MISSING VALUES")
    print("=" * 90)
    missing = pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing %": (df.isna().mean() * 100).round(2)
    }).sort_values("Missing Count", ascending=False)
    display(missing[missing["Missing Count"] > 0])



TRANSACTIONS — MISSING VALUES


,Missing Count,Missing %
mcc,2926,14.34
utr,1024,5.02



KYC — MISSING VALUES


,Missing Count,Missing %
date_of_birth,2944,8.09
monthly_income,2933,8.06
signup_timestamp,2910,7.99
aadhaar,2664,7.32
pan,1896,5.21



MERCHANTS — MISSING VALUES


,Missing Count,Missing %
settlement_account,2451,39.47
mcc,514,8.28
onboarding_date,499,8.04
declared_avg_ticket_size,371,5.97



CHARGEBACKS — MISSING VALUES


,Missing Count,Missing %


In [7]:
# CELL 7: Exact duplicate row audit
for name, df in datasets.items():
    print(f"{name:15} | Total rows: {len(df):,} | Exact duplicate rows: {df.duplicated().sum():,}")


Transactions    | Total rows: 20,400 | Exact duplicate rows: 400
KYC             | Total rows: 36,400 | Exact duplicate rows: 278
Merchants       | Total rows: 6,210 | Exact duplicate rows: 12
Chargebacks     | Total rows: 2,884 | Exact duplicate rows: 84


In [8]:
# CELL 8: Categorical profiling
for name, df in datasets.items():
    print("\n" + "=" * 90)
    print(f"{name.upper()} — CATEGORICAL VALUES")
    print("=" * 90)
    for col in df.select_dtypes(include=["object", "category"]).columns:
        print(f"\n{col}:")
        display(df[col].value_counts(dropna=False).head(30))



TRANSACTIONS — CATEGORICAL VALUES

txn_id:


txn_id
TXN00011331    2
TXN00003417    2
TXN00014122    2
TXN00006403    2
TXN00011885    2
TXN00005631    2
TXN00005044    2
TXN00016773    2
TXN00007859    2
TXN00013678    2
TXN00001946    2
TXN00015071    2
TXN00018878    2
TXN00017472    2
TXN00014620    2
TXN00018570    2
TXN00011363    2
TXN00001408    2
TXN00011408    2
TXN00015301    2
TXN00017012    2
TXN00001675    2
TXN00008489    2
TXN00015599    2
TXN00016743    2
TXN00004462    2
TXN00005461    2
TXN00001934    2
TXN00004655    2
TXN00011630    2
Name: count, dtype: int64


timestamp:


timestamp
2026/03/14    19
2026/01/14    18
2026/01/20    17
2026/03/23    17
2026/03/17    17
2026/02/09    17
2026/02/26    16
2026/01/28    16
2026/01/10    16
2026/01/24    16
2026/03/07    16
2026/03/22    15
2026/01/16    15
2026/01/30    15
2026/03/08    15
2026/01/15    15
2026/02/16    14
2026/03/30    14
2026/01/07    14
2026/03/16    14
2026/03/12    14
2026/02/10    14
2026/03/10    14
2026/02/20    14
2026/03/02    13
2026/02/04    13
2026/01/13    13
2026/01/08    13
2026/02/15    13
2026/03/01    13
Name: count, dtype: int64


user_id:


...

bank_response_timestamp
               718
04/03/2026      10
13-Feb-2026      9
03-25-2026       9
30/03/2026       9
25/03/2026       9
20/04/2026       9
2026/03/30       8
04/02/2026       8
13/02/2026       8
25-Mar-2026      8
19/03/2026       8
2026/02/14       8
21-Mar-2026      8
21/02/2026       8
03-06-2026       8
03-23-2026       8
2026/01/27       8
17/02/2026       8
2026/02/13       7
07/04/2026       7
2026/02/09       7
03-17-2026       7
27-Feb-2026      7
04-12-2026       7
07/03/2026       7
20-Feb-2026      7
21/03/2026       7
03/03/2026       7
07-Apr-2026      7
Name: count, dtype: int64


severity:


severity
P3          272
MEDIUM      268
M           261
Medium      254
LOW         250
P4          249
L           248
Low         241
P2          167
H           158
High        151
HIGH        146
P1           65
CRIT         62
Critical     52
CRITICAL     40
Name: count, dtype: int64


channel:


channel
Email          382
Branch         376
ivr            367
CHATBOT        367
IVR            362
chatbot        358
App            355
Call Center    317
Name: count, dtype: int64

In [9]:
# CELL 9: ID sample audit
id_columns = {
    "Transactions": ["txn_id", "user_id", "merchant_id"],
    "KYC": ["user_id"],
    "Merchants": ["merchant_id"],
    "Chargebacks": ["complaint_id", "txn_id", "user_id", "merchant_id"]
}

for name, cols in id_columns.items():
    df = datasets[name]
    print("\n" + "=" * 90)
    print(name.upper())
    print("=" * 90)
    for col in cols:
        if col in df.columns:
            print(f"\n{col}:")
            display(df[col].drop_duplicates().head(20))



TRANSACTIONS

txn_id:


0     TXN00011869
1     TXN00010383
2     TXN00008297
3     TXN00006448
4     TXN00018792
5     TXN00000400
6     TXN00015249
7     TXN00007121
8     TXN00008460
9     TXN00002541
10    TXN00003797
11    TXN00000092
12    TXN00004622
13    TXN00001542
14    TXN00002592
15    TXN00017665
16    TXN00002012
17    TXN00015655
18    TXN00011091
19    TXN00008584
Name: txn_id, dtype: object


user_id:


0     USR45826
1     USR79397
2     USR87810
3     USR54287
4     USR53865
5     USR90546
6     USR18691
7     USR16629
8     USR20899
9     USR51755
10    USR81003
11    USR70687
12    USR37834
13    USR78386
14    USR21114
15    USR96222
16    USR13976
17    USR55935
18    USR13890
19    USR15778
Name: user_id, dtype: object


merchant_id:


...

0     TXN00004325
1     TXN00003720
2     TXN00012539
3     TXN00017802
4     TXN00015944
5     TXN00009741
6     TXN00015086
7     TXN00014426
8     TXN00015316
9     TXN00002822
10    TXN00002456
11    TXN00015627
12    TXN00009870
13    TXN00013141
14    TXN00012165
15    TXN00019737
16    TXN00011289
17    TXN00006253
18    TXN00010961
19    TXN00005764
Name: txn_id, dtype: object


user_id:


0      usr97580
1      USR54113
2      USR17980
3      USR76148
4      USR24660
5      USR40631
6      USR58789
7      usr30157
8         86750
9      USR11987
10     USR89588
11     USR35882
12    USR 79046
13     USR22644
14     USR98915
15     USR27262
16     USR94570
17     USR19220
18     USR55439
19     USR38028
Name: user_id, dtype: object


merchant_id:


0      mch1127
1         3835
2      mch3700
3      MCH4534
4      MCH1686
5      MCH9584
6      MCH6008
7     MCH-4526
8      MCH5980
9      MCH3992
10     mch2614
11     MCH8913
12        8457
13     MCH9193
14     MCH6399
15    MCH-4852
16     MCH4545
17     MCH7286
18     MCH7581
19     MCH6758
Name: merchant_id, dtype: object

In [10]:
# CELL 10: ID format classification
def classify_id(value):
    if pd.isna(value):
        return "MISSING"
    value = str(value).strip()
    if value == "":
        return "EMPTY"
    if re.fullmatch(r"[A-Za-z0-9]+", value):
        return "ALPHANUMERIC"
    if re.search(r"\s", value):
        return "CONTAINS_SPACE"
    if re.search(r"[-_]", value):
        return "CONTAINS_SEPARATOR"
    return "OTHER"

for name, cols in id_columns.items():
    df = datasets[name]
    print("\n" + "=" * 90)
    print(f"{name.upper()} — ID FORMAT AUDIT")
    print("=" * 90)
    for col in cols:
        if col in df.columns:
            print(f"\n{col}:")
            display(df[col].map(classify_id).value_counts(dropna=False))



TRANSACTIONS — ID FORMAT AUDIT

txn_id:


txn_id
ALPHANUMERIC    20400
Name: count, dtype: int64


user_id:


user_id
ALPHANUMERIC    20400
Name: count, dtype: int64


merchant_id:


...

txn_id
ALPHANUMERIC          2697
CONTAINS_SEPARATOR     106
EMPTY                   81
Name: count, dtype: int64


user_id:


user_id
ALPHANUMERIC          2247
CONTAINS_SEPARATOR     370
CONTAINS_SPACE         267
Name: count, dtype: int64


merchant_id:


merchant_id
ALPHANUMERIC          2437
CONTAINS_SPACE         233
CONTAINS_SEPARATOR     214
Name: count, dtype: int64

In [11]:
# CELL 11: Amount format classification
def classify_amount_format(value):
    if pd.isna(value):
        return "MISSING"
    value = str(value).strip()
    if value == "":
        return "EMPTY"
    if re.search(r"(₹|INR|Rs\.?|RS\.?)", value, flags=re.IGNORECASE):
        return "CURRENCY_FORMAT"
    if re.fullmatch(r"-?\d+(?:\.\d+)?\s*[kK]", value):
        return "THOUSANDS_K_FORMAT"
    if re.fullmatch(r"-?\d+(?:\.\d+)?", value.replace(",", "")):
        if value.replace(",", "").startswith("-"):
            return "NEGATIVE_NUMERIC"
        return "NUMERIC"
    return "OTHER"

amount_columns = {
    "Transactions": ["amount"],
    "KYC": ["monthly_income"],
    "Merchants": ["declared_avg_ticket_size"],
    "Chargebacks": ["disputed_amount"]
}

for name, cols in amount_columns.items():
    for col in cols:
        if col in datasets[name].columns:
            print(f"\n{name} — {col}")
            display(datasets[name][col].map(classify_amount_format).value_counts(dropna=False))



Transactions — amount


amount
NUMERIC             12797
CURRENCY_FORMAT      7174
NEGATIVE_NUMERIC      429
Name: count, dtype: int64


KYC — monthly_income


monthly_income
NUMERIC               16217
CURRENCY_FORMAT       11170
MISSING                2933
THOUSANDS_K_FORMAT     2834
OTHER                  1790
NEGATIVE_NUMERIC       1456
Name: count, dtype: int64


Merchants — declared_avg_ticket_size


declared_avg_ticket_size
NUMERIC             2792
CURRENCY_FORMAT     2546
NEGATIVE_NUMERIC     501
MISSING              371
Name: count, dtype: int64


Chargebacks — disputed_amount


disputed_amount
NUMERIC             1356
CURRENCY_FORMAT     1114
NEGATIVE_NUMERIC     231
EMPTY                183
Name: count, dtype: int64

In [12]:
# CELL 12: Suspicious amount examples
for name, cols in amount_columns.items():
    for col in cols:
        if col in datasets[name].columns:
            classes = datasets[name][col].map(classify_amount_format)
            suspicious = datasets[name].loc[
                classes.isin(["OTHER", "NEGATIVE_NUMERIC", "EMPTY", "MISSING"]), [col]
            ].drop_duplicates().head(30)
            print(f"\n{name} — {col}")
            display(suspicious)



Transactions — amount


,amount
18,-23820.57
110,-4973.8
174,-12354.43
201,-17176.94
338,-845.26
346,-10780.52
357,-10972.41
386,-1314.1
391,-14334.84
413,-24028.5



KYC — monthly_income


,monthly_income
9,-8083
16,-81093
25,-23591
32,-28593
37,-42360
42,NaN
44,-19266
64,-23328
72,Not Available
79,-30797



Merchants — declared_avg_ticket_size


,declared_avg_ticket_size
3,-1271.48
7,-760.41
10,NaN
19,-528.91
41,-671.62
57,-698.53
60,-535.7
68,-3362.42
80,-827.01
96,-649.49



Chargebacks — disputed_amount


,disputed_amount
0,
47,-956.46
67,"-1,411.26"
101,"-2,477.45"
115,"-3,415.30"
116,"-7,727.83"
125,-759.33
136,-344.33
138,"-1,603.64"
160,-447.83


In [13]:
# CELL 13: Timestamp format classification
def classify_timestamp_format(value):
    if pd.isna(value):
        return "MISSING"
    value = str(value).strip()
    if value == "":
        return "EMPTY"
    if re.fullmatch(r"\d{10}", value):
        return "UNIX_SECONDS"
    if re.fullmatch(r"\d{13}", value):
        return "UNIX_MILLISECONDS"
    if re.search(r"[A-Za-z]{3,}", value):
        return "MONTH_NAME_OR_TEXT"
    if "/" in value:
        return "SLASH_DATE"
    if "-" in value:
        return "HYPHEN_DATE"
    if re.fullmatch(r"\d+(?:\.\d+)?", value):
        return "NUMERIC_TIMESTAMP"
    return "OTHER"

for name, df in datasets.items():
    timestamp_cols = [
        c for c in df.columns
        if any(t in c.lower() for t in ["date", "time", "timestamp", "created", "reported", "response"])
    ]
    print("\n" + "=" * 90)
    print(name.upper())
    print("=" * 90)
    for col in timestamp_cols:
        print(f"\n{col}:")
        display(df[col].map(classify_timestamp_format).value_counts(dropna=False))



TRANSACTIONS

timestamp:


timestamp
HYPHEN_DATE     14335
SLASH_DATE       5053
UNIX_SECONDS     1012
Name: count, dtype: int64


KYC

date_of_birth:


date_of_birth
SLASH_DATE            16090
HYPHEN_DATE           10079
MONTH_NAME_OR_TEXT     5033
MISSING                2944
NUMERIC_TIMESTAMP      1852
UNIX_SECONDS            402
Name: count, dtype: int64


signup_timestamp:


...

transaction_timestamp
SLASH_DATE            1267
HYPHEN_DATE            741
MONTH_NAME_OR_TEXT     408
EMPTY                  235
UNIX_SECONDS           233
Name: count, dtype: int64


reported_timestamp:


reported_timestamp
SLASH_DATE            1336
HYPHEN_DATE            690
MONTH_NAME_OR_TEXT     412
UNIX_SECONDS           237
EMPTY                  209
Name: count, dtype: int64


bank_response_timestamp:


bank_response_timestamp
SLASH_DATE            1064
EMPTY                  718
HYPHEN_DATE            594
MONTH_NAME_OR_TEXT     353
UNIX_SECONDS           155
Name: count, dtype: int64

In [14]:
# CELL 14: Suspicious timestamp examples
for name, df in datasets.items():
    timestamp_cols = [
        c for c in df.columns
        if any(t in c.lower() for t in ["date", "time", "timestamp", "created", "reported", "response"])
    ]
    for col in timestamp_cols:
        classes = df[col].map(classify_timestamp_format)
        suspicious = df.loc[classes.isin(["OTHER", "EMPTY", "MISSING"]), [col]].drop_duplicates().head(30)
        if len(suspicious):
            print(f"\n{name} — {col}")
            display(suspicious)



KYC — date_of_birth


,date_of_birth
1,NaN



KYC — signup_timestamp


,signup_timestamp
6,NaN



Merchants — onboarding_date


...

,transaction_timestamp
40,



Chargebacks — reported_timestamp


,reported_timestamp
19,



Chargebacks — bank_response_timestamp


,bank_response_timestamp
12,


In [15]:
# CELL 15: Temporary mixed timestamp parser
def parse_mixed_timestamp(value):
    if pd.isna(value):
        return pd.NaT
    text = str(value).strip()
    if text == "":
        return pd.NaT
    if re.fullmatch(r"\d{10}", text):
        return pd.to_datetime(int(text), unit="s", errors="coerce")
    if re.fullmatch(r"\d{13}", text):
        return pd.to_datetime(int(text), unit="ms", errors="coerce")
    return pd.to_datetime(text, errors="coerce")

for name, df in datasets.items():
    timestamp_cols = [
        c for c in df.columns
        if any(t in c.lower() for t in ["date", "time", "timestamp", "created", "reported", "response"])
    ]
    for col in timestamp_cols:
        parsed = df[col].map(parse_mixed_timestamp)
        print(f"{name:15} | {col:25} | Parsed: {parsed.notna().sum():,} | Failed: {parsed.isna().sum():,}")


C:\Users\HP\AppData\Local\Temp\ipykernel_23864\3939992099.py:12: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(text, errors="coerce")


Transactions    | timestamp                 | Parsed: 20,400 | Failed: 0


C:\Users\HP\AppData\Local\Temp\ipykernel_23864\3939992099.py:12: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(text, errors="coerce")
C:\Users\HP\AppData\Local\Temp\ipykernel_23864\3939992099.py:12: UserWarning: Parsing dates in %d/%m/%Y %I:%M %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(text, errors="coerce")


KYC             | date_of_birth             | Parsed: 31,027 | Failed: 5,373
KYC             | signup_timestamp          | Parsed: 33,490 | Failed: 2,910
Merchants       | onboarding_date           | Parsed: 5,711 | Failed: 499
Chargebacks     | transaction_timestamp     | Parsed: 2,649 | Failed: 235
Chargebacks     | reported_timestamp        | Parsed: 2,675 | Failed: 209
Chargebacks     | bank_response_timestamp   | Parsed: 2,166 | Failed: 718


In [16]:
# CELL 16: Canonical ID normalization
def normalize_id(value, prefix):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().upper()
    value = re.sub(r"[\s\-_]", "", value)
    value = re.sub(rf"^{prefix}", "", value)
    value = re.sub(r"[^A-Z0-9]", "", value)
    if value == "":
        return pd.NA
    return f"{prefix}{value}"

tx_profile = transactions.copy()
kyc_profile = kyc.copy()
merchant_profile = merchants.copy()
cb_profile = chargebacks.copy()

tx_profile["user_id_norm"] = tx_profile["user_id"].map(lambda x: normalize_id(x, "USR"))
tx_profile["merchant_id_norm"] = tx_profile["merchant_id"].map(lambda x: normalize_id(x, "MCH"))
tx_profile["txn_id_norm"] = tx_profile["txn_id"].map(lambda x: normalize_id(x, "TXN"))

kyc_profile["user_id_norm"] = kyc_profile["user_id"].map(lambda x: normalize_id(x, "USR"))
merchant_profile["merchant_id_norm"] = merchant_profile["merchant_id"].map(lambda x: normalize_id(x, "MCH"))

cb_profile["user_id_norm"] = cb_profile["user_id"].map(lambda x: normalize_id(x, "USR"))
cb_profile["merchant_id_norm"] = cb_profile["merchant_id"].map(lambda x: normalize_id(x, "MCH"))
cb_profile["txn_id_norm"] = cb_profile["txn_id"].map(lambda x: normalize_id(x, "TXN"))

print("Canonical ID normalization completed.")


Canonical ID normalization completed.


In [17]:
# CELL 17: Join coverage audit
def join_coverage(source, source_key, target, target_key):
    target_values = set(target[target_key].dropna().astype(str))
    matched = source[source_key].astype("string").isin(target_values)
    return {
        "Total": len(source),
        "Matched": int(matched.sum()),
        "Unmatched": int((~matched).sum()),
        "Match %": round(matched.mean() * 100, 2)
    }

join_results = {
    "Transactions → KYC": join_coverage(tx_profile, "user_id_norm", kyc_profile, "user_id_norm"),
    "Transactions → Merchants": join_coverage(tx_profile, "merchant_id_norm", merchant_profile, "merchant_id_norm"),
    "Chargebacks → Transactions": join_coverage(cb_profile, "txn_id_norm", tx_profile, "txn_id_norm"),
    "Chargebacks → KYC": join_coverage(cb_profile, "user_id_norm", kyc_profile, "user_id_norm"),
    "Chargebacks → Merchants": join_coverage(cb_profile, "merchant_id_norm", merchant_profile, "merchant_id_norm")
}

display(pd.DataFrame(join_results).T)


,Total,Matched,Unmatched,Match %
Transactions → KYC,"20,400.00","6,617.00","13,783.00",32.44
Transactions → Merchants,"20,400.00","9,809.00","10,591.00",48.08
Chargebacks → Transactions,"2,884.00","2,683.00",201.00,93.03
Chargebacks → KYC,"2,884.00",916.00,"1,968.00",31.76
Chargebacks → Merchants,"2,884.00","1,336.00","1,548.00",46.32


In [18]:
# CELL 18: Unmatched canonical ID examples
pairs = [
    ("Transactions → KYC", tx_profile, "user_id_norm", kyc_profile, "user_id_norm"),
    ("Transactions → Merchants", tx_profile, "merchant_id_norm", merchant_profile, "merchant_id_norm"),
    ("Chargebacks → Transactions", cb_profile, "txn_id_norm", tx_profile, "txn_id_norm"),
    ("Chargebacks → KYC", cb_profile, "user_id_norm", kyc_profile, "user_id_norm"),
    ("Chargebacks → Merchants", cb_profile, "merchant_id_norm", merchant_profile, "merchant_id_norm")
]

for name, source, sk, target, tk in pairs:
    target_values = set(target[tk].dropna().astype(str))
    unmatched = source.loc[~source[sk].astype("string").isin(target_values), sk].dropna().drop_duplicates().head(20)
    print(f"\n{name}")
    display(unmatched)



Transactions → KYC


0     USR45826
2     USR87810
3     USR54287
4     USR53865
7     USR16629
9     USR51755
10    USR81003
12    USR37834
13    USR78386
14    USR21114
15    USR96222
16    USR13976
17    USR55935
18    USR13890
19    USR15778
20    USR15834
21    USR33189
23    USR99474
25    USR85411
27    USR54536
Name: user_id_norm, dtype: object


Transactions → Merchants


0     MCH7045
1     MCH5031
2     MCH9809
4     MCH8121
7     MCH7753
11    MCH4523
13    MCH4890
14    MCH9819
16    MCH8222
18    MCH8216
21    MCH4645
22    MCH4207
23    MCH1206
25    MCH6621
26    MCH8759
28    MCH5711
29    MCH9490
31    MCH1798
35    MCH5104
36    MCH3197
Name: merchant_id_norm, dtype: object


Chargebacks → Transactions


29     TXN65742
42     TXN66039
59     TXN84758
67     TXN94213
95     TXN40959
98     TXN61071
147    TXN95161
195    TXN54083
198    TXN56695
205    TXN37348
216    TXN88540
253    TXN88490
298    TXN84777
343    TXN94657
361    TXN81951
369    TXN34155
380    TXN71282
419    TXN86247
464    TXN46623
466    TXN96928
Name: txn_id_norm, dtype: object


Chargebacks → KYC


1     USR54113
2     USR17980
3     USR76148
4     USR24660
5     USR40631
7     USR30157
8     USR86750
10    USR89588
11    USR35882
14    USR98915
16    USR94570
17    USR19220
23    USR67471
25    USR79443
27    USR73891
29    USR32980
30    USR41524
31    USR11727
33    USR92751
35    USR27888
Name: user_id_norm, dtype: object


Chargebacks → Merchants


0     MCH1127
6     MCH6008
9     MCH3992
10    MCH2614
11    MCH8913
12    MCH8457
14    MCH6399
15    MCH4852
16    MCH4545
17    MCH7286
22    MCH3957
23    MCH8738
25    MCH3729
26    MCH9105
30    MCH2342
34    MCH7438
35    MCH5076
37    MCH4183
38    MCH2259
39    MCH4252
Name: merchant_id_norm, dtype: object

In [19]:
# CELL 19: Key uniqueness audit
key_specs = [
    ("Transactions", tx_profile, "txn_id"),
    ("Transactions", tx_profile, "user_id"),
    ("Transactions", tx_profile, "merchant_id"),
    ("KYC", kyc_profile, "user_id"),
    ("Merchants", merchant_profile, "merchant_id"),
    ("Chargebacks", cb_profile, "complaint_id"),
    ("Chargebacks", cb_profile, "txn_id"),
    ("Chargebacks", cb_profile, "user_id"),
    ("Chargebacks", cb_profile, "merchant_id")
]

audit = []
for dataset_name, df, key in key_specs:
    mask = df[key].duplicated(keep=False)
    audit.append({
        "Key": f"{dataset_name} - {key}",
        "Total": len(df),
        "Non-Null": int(df[key].notna().sum()),
        "Unique Values": int(df[key].nunique(dropna=True)),
        "Duplicated Rows": int(mask.sum()),
        "Duplicated Keys": int(df.loc[mask, key].nunique())
    })

display(pd.DataFrame(audit))


,Key,Total,Non-Null,Unique Values,Duplicated Rows,Duplicated Keys
0,Transactions - txn_id,20400,20400,20000,800,400
1,Transactions - user_id,20400,20400,17878,4786,2264
2,Transactions - merchant_id,20400,20400,8051,18288,5939
3,KYC - user_id,36400,36400,32165,8053,3818
4,Merchants - merchant_id,6210,6210,5083,2049,922
5,Chargebacks - complaint_id,2884,2884,2800,168,84
6,Chargebacks - txn_id,2884,2884,2582,509,207
7,Chargebacks - user_id,2884,2884,2454,648,218
8,Chargebacks - merchant_id,2884,2884,2051,1079,246


In [20]:
# CELL 20: Duplicate key examples
for dataset_name, df, key in key_specs:
    duplicate_keys = df.loc[df[key].duplicated(keep=False), key].dropna().unique()[:5]
    if len(duplicate_keys):
        print("\n" + "=" * 90)
        print(f"{dataset_name} — {key}")
        print("=" * 90)
        display(df[df[key].isin(duplicate_keys)].sort_values(key).head(20))



Transactions — txn_id


,txn_id,timestamp,user_id,merchant_id,amount,utr,mcc,status,user_id_norm,merchant_id_norm,txn_id_norm
113,TXN00005885,02-05-2026 06:19:19 PM,USR17338,MCH5199,"₹12,009.79",UTR6613055314,NaN,TXN_SUCCESS,USR17338,MCH5199,TXN00005885
16930,TXN00005885,02-05-2026 06:19:19 PM,USR17338,MCH5199,"₹12,009.79",UTR6613055314,NaN,TXN_SUCCESS,USR17338,MCH5199,TXN00005885
69,TXN00005968,2026/02/20,USR20374,MCH9694,"INR 2,195",UTR0092134543,"5,411.00",TXN_SUCCESS,USR20374,MCH9694,TXN00005968
16776,TXN00005968,2026/02/20,USR20374,MCH9694,"INR 2,195",UTR0092134543,"5,411.00",TXN_SUCCESS,USR20374,MCH9694,TXN00005968
125,TXN00006562,2026-03-22 13:46:10,USR64443,MCH7071,Rs. 8094.88,UTR1599752498,NaN,COMPLETED,USR64443,MCH7071,TXN00006562
567,TXN00006562,2026-03-22 13:46:10,USR64443,MCH7071,Rs. 8094.88,UTR1599752498,NaN,COMPLETED,USR64443,MCH7071,TXN00006562
2,TXN00008297,2026-01-23 14:10:16,USR87810,MCH9809,15446.19,UTR9037001889,"5,411.00",S,USR87810,MCH9809,TXN00008297
7672,TXN00008297,2026-01-23 14:10:16,USR87810,MCH9809,15446.19,UTR9037001889,"5,411.00",S,USR87810,MCH9809,TXN00008297
153,TXN00013932,15/02/2026 01:49:16,USR12186,MCH9903,Rs. 22356.76,UTR2777053913,"7,011.00",SUCCESS,USR12186,MCH9903,TXN00013932
20248,TXN00013932,15/02/2026 01:49:16,USR12186,MCH9903,Rs. 22356.76,UTR2777053913,"7,011.00",SUCCESS,USR12186,MCH9903,TXN00013932



Transactions — user_id


,txn_id,timestamp,user_id,merchant_id,amount,utr,mcc,status,user_id_norm,merchant_id_norm,txn_id_norm
18,TXN00011091,03/03/2026 19:17:39,USR13890,MCH8216,-23820.57,UTR6527062794,"4,131.00",COMPLETED,USR13890,MCH8216,TXN00011091
14972,TXN00007080,2026-03-24 17:53:20,USR13890,MCH3301,18460.0,UTR1947736910,"4,131.00",S,USR13890,MCH3301,TXN00007080
7,TXN00007121,25/02/2026 00:53:02,USR16629,MCH7753,Rs. 22687.5,UTR3330570580,"5,411.00",COMPLETED,USR16629,MCH7753,TXN00007121
4623,TXN00014141,03-01-2026 12:21:48 AM,USR16629,MCH5175,"₹8,742.13",UTR9544808491,"5,411.00",Success,USR16629,MCH5175,TXN00014141
3,TXN00006448,1770063471,USR54287,MCH6928,12110.49,UTR5257823698,"5,411.00",S,USR54287,MCH6928,TXN00006448
11682,TXN00006303,01-04-2026 07:49:19 PM,USR54287,MCH7397,12136.82,UTR5459477460,"4,131.00",SUCCESS,USR54287,MCH7397,TXN00006303
2,TXN00008297,2026-01-23 14:10:16,USR87810,MCH9809,15446.19,UTR9037001889,"5,411.00",S,USR87810,MCH9809,TXN00008297
7672,TXN00008297,2026-01-23 14:10:16,USR87810,MCH9809,15446.19,UTR9037001889,"5,411.00",S,USR87810,MCH9809,TXN00008297
5,TXN00000400,2026-02-11 01:40:43,USR90546,MCH6773,Rs. 18866.03,UTR2080442730,"5,812.00",SUCCESS,USR90546,MCH6773,TXN00000400
17879,TXN00013152,2026-03-02 13:04:07,USR90546,MCH6412,"₹1,140.29",UTR5260223609,"4,131.00",COMPLETED,USR90546,MCH6412,TXN00013152



Transactions — merchant_id


...

,complaint_id,txn_id,user_id,merchant_id,transaction_timestamp,reported_timestamp,disputed_amount,reason_code,complaint_text,resolution_status,bank_response_timestamp,severity,channel,user_id_norm,merchant_id_norm,txn_id_norm
9,CBK0002541,TXN00002822,USR11987,MCH3992,2026-03-18 10:04:06,23/03/2026,,UNAUTHORISED,User claims account was compromised before tra...,OPEN,30/03/2026,P2,ivr,USR11987,MCH3992,TXN00002822
2200,CBK0002541,TXN00002822,USR11987,MCH3992,2026-03-18 10:04:06,23/03/2026,,UNAUTHORISED,User claims account was compromised before tra...,OPEN,30/03/2026,P2,ivr,USR11987,MCH3992,TXN00002822
1,CBK0001941,TXN00003720,USR54113,3835,25/02/2026 10:24 AM,1772691855,414.69,login compromised,User reports money deducted but merchant denie...,In Progress,12/03/2026 06:24 AM,H,Call Center,USR54113,MCH3835,TXN00003720
1860,CBK0001941,TXN00003720,USR54113,3835,25/02/2026 10:24 AM,1772691855,414.69,login compromised,User reports money deducted but merchant denie...,In Progress,12/03/2026 06:24 AM,H,Call Center,USR54113,MCH3835,TXN00003720
23,CBK0001330,TXN00005940,usr67471,MCH8738,14/03/2026 12:34 AM,15/03/2026 12:34 PM,"INR 1,360.01",DUP_DEBIT,Suspicious high-value payment disputed by cust...,Pending Bank,03-27-2026,MEDIUM,CHATBOT,USR67471,MCH8738,TXN00005940
2805,CBK0001330,TXN00005940,usr67471,MCH8738,14/03/2026 12:34 AM,15/03/2026 12:34 PM,"INR 1,360.01",DUP_DEBIT,Suspicious high-value payment disputed by cust...,Pending Bank,03-27-2026,MEDIUM,CHATBOT,USR67471,MCH8738,TXN00005940
16,CBK0001099,TXN00011289,USR94570,MCH4545,30/01/2026 04:49 AM,02-04-2026,Rs. 305,Account Takeover,Payment made to unknown merchant as per custom...,Closed,02-28-2026,CRITICAL,IVR,USR94570,MCH4545,TXN00011289
796,CBK0001762,TXN00011289,USR14871,MCH4080,2026/02/19,27-Feb-2026,"1,035.00",item not received,User reports money deducted but merchant denie...,WIP,2026-03-22 04:25:49,High,Branch,USR14871,MCH4080,TXN00011289
21,CBK0000512,TXN00014054,usr_27947,MCH7667,21-Jan-2026,01-26-2026,"4,010.97",Unauthorized Transaction,Customer says amount was debited twice.,Closed,01-30-2026,P4,CHATBOT,USR27947,MCH7667,TXN00014054
2465,CBK0001201,TXN00014054,usr85014,MCH-7286,15/03/2026,2026/03/15,,Merchant Not Delivered,Complaint received after multiple failed attem...,Closed,27/03/2026 08:14 AM,LOW,CHATBOT,USR85014,MCH7286,TXN00014054



Chargebacks — user_id


,complaint_id,txn_id,user_id,merchant_id,transaction_timestamp,reported_timestamp,disputed_amount,reason_code,complaint_text,resolution_status,bank_response_timestamp,severity,channel,user_id_norm,merchant_id_norm,txn_id_norm
2579,CBK0002265,TXN00008431,USR11987,MCH5922,,2026-02-07 23:51:35,"1,831.96",amount mismatch,Customer says transaction was not authorized.,REJECTED,09/03/2026 11:51 PM,P3,CHATBOT,USR11987,MCH5922,TXN00008431
2200,CBK0002541,TXN00002822,USR11987,MCH3992,2026-03-18 10:04:06,23/03/2026,,UNAUTHORISED,User claims account was compromised before tra...,OPEN,30/03/2026,P2,ivr,USR11987,MCH3992,TXN00002822
1924,CBK0001679,TXN00007437,USR11987,MCH2576,25-Mar-2026,2026-03-27 17:04:00,"₹1,171.85",login compromised,Customer says transaction was not authorized.,Rejected,04-21-2026,MEDIUM,ivr,USR11987,MCH2576,TXN00007437
9,CBK0002541,TXN00002822,USR11987,MCH3992,2026-03-18 10:04:06,23/03/2026,,UNAUTHORISED,User claims account was compromised before tra...,OPEN,30/03/2026,P2,ivr,USR11987,MCH3992,TXN00002822
65,CBK0002156,TXN00013206,USR11987,MCH 6665,24/01/2026 09:52 AM,25/01/2026 04:52 AM,"26,477.69",Customer Dispute,Complaint received after multiple failed attem...,In Progress,,High,Email,USR11987,MCH6665,TXN00013206
1522,CBK0001626,TXN49458,USR11987,MCH8783,20-Feb-2026,22/02/2026 06:45 PM,"12,414.63",Customer Dispute,User claims account was compromised before tra...,Open,03/03/2026,CRIT,Branch,USR11987,MCH8783,TXN49458
1337,CBK0000462,TXN00018761,USR24660,mch9291,2026/01/23,2026/01/24,"INR 1,500.59",unauthorized_transaction,Customer says amount was debited twice.,IN_PROGRESS,17-Feb-2026,L,Email,USR24660,MCH9291,TXN00018761
2664,CBK0000939,TXN00009825,USR24660,MCH9618,2026/03/26,29/03/2026 06:03 AM,"Rs. 3,080",customer issue,Complaint received after multiple failed attem...,Resolved,25/04/2026,P4,Branch,USR24660,MCH9618,TXN00009825
2833,CBK0001376,TXN00009369,USR24660,4473,2026/02/16,19/02/2026,"2,070.15",Account Takeover,Customer complaint text unclear due to poor ca...,RESOLVED,03-06-2026,P4,Branch,USR24660,MCH4473,TXN00009369
687,CBK0001833,txn-00009923,USR24660,MCH-3434,04/01/2026 12:15 PM,2025-12-30 12:15:20,"Rs. 2,940",complaint,Merchant service was not delivered after payment.,Open,12-31-2025,Medium,Call Center,USR24660,MCH3434,TXN00009923



Chargebacks — merchant_id


,complaint_id,txn_id,user_id,merchant_id,transaction_timestamp,reported_timestamp,disputed_amount,reason_code,complaint_text,resolution_status,bank_response_timestamp,severity,channel,user_id_norm,merchant_id_norm,txn_id_norm
1,CBK0001941,TXN00003720,USR54113,3835,25/02/2026 10:24 AM,1772691855,414.69,login compromised,User reports money deducted but merchant denie...,In Progress,12/03/2026 06:24 AM,H,Call Center,USR54113,MCH3835,TXN00003720
1860,CBK0001941,TXN00003720,USR54113,3835,25/02/2026 10:24 AM,1772691855,414.69,login compromised,User reports money deducted but merchant denie...,In Progress,12/03/2026 06:24 AM,H,Call Center,USR54113,MCH3835,TXN00003720
2615,CBK0000429,TXN00019661,USR97772,MCH3992,19/02/2026 10:56 PM,2026-02-21 09:56:09,,amount mismatch,User reports money deducted but merchant denie...,Rejected,06-Mar-2026,P2,App,USR97772,MCH3992,TXN00019661
2512,CBK0002296,TXN00004293,USR72591,MCH3992,01-05-2026,01-08-2026,830.77,item not received,Complaint received after multiple failed attem...,Closed,14-Jan-2026,MEDIUM,Branch,USR72591,MCH3992,TXN00004293
2478,CBK0000363,TXN00014836,USR62092,MCH3992,22-Mar-2026,06/04/2026,,dispute raised,user reports money deducted but merchant denie...,RESOLVED,28/04/2026 12:09 PM,L,chatbot,USR62092,MCH3992,TXN00014836
2285,CBK0000942,TXN00000964,usr_78941,MCH3992,1774797497,06/04/2026,"2,761.55",charged twice,Suspicious high-value payment disputed by cust...,RESOLVED,06/04/2026 02:18 AM,L,ivr,USR78941,MCH3992,TXN00000964
2204,CBK0000175,TXN00009098,USR 33215,MCH3992,1767486218,01/01/2026 12:23 AM,"4,947.89",service failed,Payment made to unknown merchant as per custom...,CLOSED,28-Jan-2026,Low,ivr,USR33215,MCH3992,TXN00009098
2200,CBK0002541,TXN00002822,USR11987,MCH3992,2026-03-18 10:04:06,23/03/2026,,UNAUTHORISED,User claims account was compromised before tra...,OPEN,30/03/2026,P2,ivr,USR11987,MCH3992,TXN00002822
1064,CBK0000411,TXN00003133,USR48609,MCH3992,,03-20-2026,"3,287.47",unauthorized_transaction,complaint received after multiple failed attem...,Pending Bank,2026/04/16,Medium,Email,USR48609,MCH3992,TXN00003133
1729,CBK0002066,TXN00014413,USR86472,MCH3992,29/03/2026,2026-04-01 21:21:33,,amount mismatch,User reports money deducted but merchant denie...,WIP,,LOW,ivr,USR86472,MCH3992,TXN00014413


In [21]:
# CELL 21: Chargeback-to-transaction linkage quality
tx_ids = set(tx_profile["txn_id_norm"].dropna().astype(str))
matched = cb_profile["txn_id_norm"].astype("string").isin(tx_ids)

print("Chargeback records:", len(cb_profile))
print("Matched to transactions:", int(matched.sum()))
print("Unmatched:", int((~matched).sum()))
print("Match rate:", round(matched.mean() * 100, 2), "%")

display(cb_profile.loc[~matched, ["complaint_id", "txn_id", "txn_id_norm", "user_id", "merchant_id"]].head(30))


Chargeback records: 2884
Matched to transactions: 2683
Unmatched: 201
Match rate: 93.03 %


,complaint_id,txn_id,txn_id_norm,user_id,merchant_id
29,CBK0000594,TXN65742,TXN65742,USR32980,MCH7020
42,CBK0002798,TXN66039,TXN66039,USR24087,MCH6965
56,CBK0002758,,<NA>,USR60667,MCH2383
59,CBK0001020,TXN84758,TXN84758,USR36590,mch1368
67,CBK0000432,TXN94213,TXN94213,USR96958,MCH7286
77,CBK0001968,,<NA>,USR46599,MCH1320
95,CBK0002780,TXN40959,TXN40959,USR75376,MCH 4293
98,CBK0001175,TXN61071,TXN61071,USR22543,MCH3116
107,CBK0001532,,<NA>,USR52450,MCH6244
143,CBK0001382,,<NA>,USR57424,MCH4684


In [ ]:
# CELL 22: Entity conflict audit
def conflict_summary(df, key):
    attributes = [c for c in df.columns if c != key]
    flags = []
    for entity_id, group in df.groupby(key, dropna=False):
        conflict = any(group[c].dropna().astype(str).nunique() > 1 for c in attributes)
        flags.append(conflict)
    return {
        "Unique entities": len(flags),
        "Conflicting entities": int(sum(flags)),
        "Non-conflicting entities": int(len(flags) - sum(flags))
    }

print("KYC:", conflict_summary(kyc_profile, "user_id_norm"))
print("Merchants:", conflict_summary(merchant_profile, "merchant_id_norm"))


In [ ]:
# CELL 23: Entity conflict examples
def show_conflict_examples(df, key, n=5):
    attributes = [c for c in df.columns if c != key]
    ids = []
    for entity_id, group in df.groupby(key, dropna=False):
        if pd.isna(entity_id):
            continue
        if any(group[c].dropna().astype(str).nunique() > 1 for c in attributes):
            ids.append(entity_id)
        if len(ids) >= n:
            break
    if ids:
        display(df[df[key].isin(ids)].sort_values(key))

print("KYC conflict examples:")
show_conflict_examples(kyc_profile, "user_id_norm")

print("Merchant conflict examples:")
show_conflict_examples(merchant_profile, "merchant_id_norm")


In [ ]:
# CELL 24: Business-rule / amount validation
def parse_money_for_audit(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text == "":
        return np.nan

    text = re.sub(r"₹|INR|Rs\.?|RS\.?", "", text, flags=re.IGNORECASE)
    text = text.replace(",", "").strip()

    match = re.fullmatch(r"(-?\d+(?:\.\d+)?)\s*[kK]", text)
    if match:
        return float(match.group(1)) * 1000

    return pd.to_numeric(text, errors="coerce")

tx_amount = tx_profile["amount"].map(parse_money_for_audit)
kyc_income = kyc_profile["monthly_income"].map(parse_money_for_audit)
merchant_ticket = merchant_profile["declared_avg_ticket_size"].map(parse_money_for_audit)
cb_amount = cb_profile["disputed_amount"].map(parse_money_for_audit)

print("=" * 90)
print("BUSINESS RULE / LOGICAL VALIDATION AUDIT")
print("=" * 90)
print("Transactions with amount <= 0:", int((tx_amount <= 0).sum()))
print("KYC records with income <= 0:", int((kyc_income <= 0).sum()))
print("Merchants with ticket size <= 0:", int((merchant_ticket <= 0).sum()))
print("Chargebacks with disputed amount <= 0:", int((cb_amount <= 0).sum()))


In [ ]:
# CELL 25: Chargeback timeline validation
date_cols = [
    c for c in cb_profile.columns
    if any(t in c.lower() for t in ["date", "time", "timestamp"])
]

for col in date_cols:
    cb_profile[f"{col}_parsed"] = cb_profile[col].map(parse_mixed_timestamp)

reported = next((c for c in date_cols if "reported" in c.lower()), None)
txn_date = next((c for c in date_cols if "transaction" in c.lower()), None)
response = next((c for c in date_cols if "response" in c.lower() or "bank" in c.lower()), None)

print("=" * 90)
print("CHARGEBACK TIMELINE VALIDATION")
print("=" * 90)

if reported and txn_date:
    a = cb_profile[f"{reported}_parsed"]
    b = cb_profile[f"{txn_date}_parsed"]
    flag = a.notna() & b.notna() & (a < b)
    print("Reported before transaction:", int(flag.sum()))
else:
    print("Could not automatically identify reported/transaction date columns.")

if response and reported:
    a = cb_profile[f"{response}_parsed"]
    b = cb_profile[f"{reported}_parsed"]
    flag = a.notna() & b.notna() & (a < b)
    print("Bank response before reported:", int(flag.sum()))
else:
    print("Could not automatically identify response/reported date columns.")


In [ ]:
# CELL 26: MCC quality audit
def normalize_mcc_for_audit(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().upper()
    if text in {"", "UNKNOWN", "N/A", "NA", "NONE", "NAN"}:
        return pd.NA

    text = re.sub(r"^MCC[-_\s]*", "", text)
    text = re.sub(r"\.0+$", "", text)
    digits = re.sub(r"\D", "", text)

    if len(digits) == 4:
        return digits

    # Recover values padded with leading zeros, e.g. 05311 -> 5311.
    stripped = digits.lstrip("0")
    if len(stripped) == 4:
        return stripped

    return pd.NA

for name, df in [("Transactions", tx_profile), ("Merchants", merchant_profile)]:
    if "mcc" not in df.columns:
        continue

    normalized = df["mcc"].map(normalize_mcc_for_audit)
    missing = df["mcc"].isna() | df["mcc"].astype(str).str.strip().isin(
        ["", "UNKNOWN", "N/A", "NA", "NONE", "NAN"]
    )
    malformed = (~missing) & normalized.isna()

    print("\n" + "=" * 90)
    print(f"{name.upper()} — MCC QUALITY AUDIT")
    print("=" * 90)
    print("Missing MCC:", int(missing.sum()))
    print("Malformed MCC:", int(malformed.sum()))
    print("Malformed MCC examples:")
    display(df.loc[malformed, [c for c in ["merchant_id", "mcc"] if c in df.columns]].drop_duplicates().head(30))


# Final Data Profiling & Quality Findings

## Dataset Overview

- UPI Transactions: **20,400**
- KYC Records: **36,400**
- Merchant Master: **6,210**
- Chargebacks: **2,884**

The datasets contain significant real-world quality issues. The cleaning strategy will focus on **data rescue, normalization, validation, and anomaly flagging**, rather than blindly deleting problematic records.

## Key Findings

### Identifier & Join Quality

Canonical identifier normalization was performed for `user_id`, `merchant_id`, and `txn_id`.

- Transactions → KYC: **32.44%**
- Transactions → Merchant Master: **48.08%**
- Chargebacks → Transactions: **93.03%**
- Chargebacks → KYC: **31.76%**
- Chargebacks → Merchant Master: **46.32%**

The **93.03% chargeback-to-transaction linkage** is especially important for dispute analytics.

### Duplicate & Key Integrity

- Transactions: **20,000 unique transaction IDs** across 20,400 records.
- KYC: **28,920 unique users** across 36,400 records.
- Merchants: **4,343 unique merchant IDs** across 6,210 records.
- Chargebacks: **2,800 unique complaint IDs** across 2,884 records.

KYC and Merchant Master are **not one-row-per-ID**, so they must not be blindly deduplicated by entity ID.

### Entity Conflicts

- KYC users with conflicting attributes: **6,048**
- Merchant IDs with conflicting attributes: **1,407**

These conflicts may become useful data-quality/fraud-risk signals.

### Monetary Quality

Non-positive monetary values:

- Transactions: **429**
- KYC income: **1,456**
- Merchant ticket size: **501**
- Chargebacks: **231**

Currency symbols, `INR`/`Rs.` prefixes, commas, and `k` notation were also identified.

### Timestamp Quality

Chargeback timeline violations:

- Reported before transaction: **392**
- Bank response before reported: **305**

These records will be retained with explicit anomaly flags.

### Categorical Quality

Multiple representations of the same logical value were identified across transaction status, KYC status/risk, merchant categories/business types/statuses, chargeback reasons/severity/resolution/channels, and city names.

These will be standardized during cleaning.

### MCC Quality

Recoverable formatting variations include:

- `MCC-7011` → `7011`
- `5311.0` → `5311`
- `05311` → `5311`
- `MCC-5912` → `5912`
- `05812` → `5812`

The profiling identified:

**Transactions**
- Missing MCC: **2,926**
- Malformed/non-canonical MCC before recovery: **17,474**

**Merchant Master**
- Missing MCC: **514**
- Malformed/non-canonical MCC before recovery: **1,634**

Cleaning will follow:

**Normalize → Recover → Validate → Flag unrecoverable values**

## Overall Cleaning Strategy

1. Preserve raw files.
2. Normalize identifiers.
3. Clean monetary values.
4. Standardize timestamps.
5. Normalize categorical values.
6. Recover malformed MCC values.
7. Handle exact duplicates separately from conflicting entity records.
8. Preserve unmatched joins with flags.
9. Preserve entity conflicts as potential fraud/data-quality signals.
10. Preserve timeline anomalies with flags.
11. Validate chargeback-to-transaction linkage before dispute metrics.
12. Aggregate chargebacks appropriately before joining to transaction-level analytics.

**Objective:** create a trusted analytical dataset without destroying potentially useful fraud indicators.
